# Load an MLP model and convert it to ONNX file
Please take the following steps before running this notebook
1. clone the repo by running `git clone https://github.com/abidihaider/RealTimeAlignment.git`
2. check to the develop branch of the repo
3. run `python setup.py develop`

In [1]:
import os
import sys
from pathlib import Path
import yaml
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rich import print as rprint

import torch
from torch import nn
import torch.quantization as tq

import onnx
import onnxruntime

from rtal.datasets.dataset import ROMDataset
from torch.utils.data import DataLoader

from mlp_for_quantization import MLP, QuantModel

In [2]:
onnx_folder = Path('onnx_files_narrow')

## Load some data

In [3]:
data_root = '/data/yhuang2/rtal/rom_det-3_part-200_cont-and-rounded'
dataset = ROMDataset(data_root, split='train', num_particles=50)
train_bs = 4
dataloader = DataLoader(dataset, batch_size=train_bs, shuffle=False)
print(f'Number of train batches (bs={train_bs}), {len(dataloader)}')

dataset_test = ROMDataset(data_root, split='test', num_particles=50)
test_bs = 1
dataloader_test = DataLoader(dataset_test, batch_size=test_bs, shuffle=False)
print(f'Number of test batches (bs={test_bs}), {len(dataloader_test)}')

Number of train batches (bs=4), 25000
Number of test batches (bs=1), 10000


## Load model configuration and use it to initialize models
For the half model, we will cast it to half precision for inference.

In [4]:
with open('checkpoints/config_narrow.yaml', 'r', encoding='UTF-8') as handle:
    config = yaml.safe_load(handle)

# wrap the MLP model inside a model with quant/dequant stubs
model = QuantModel(MLP(**config['model']))

In [5]:
ckpt_path = 'checkpoints/ckpt_last_narrow.pth'
ckpt = torch.load(ckpt_path, map_location='cpu')
model_state_dict = ckpt['model']

# load the pretrained weights into the model
model.model.load_state_dict(model_state_dict)
model.eval()

QuantModel(
  (quant): QuantStub()
  (model): MLP(
    (embed): Sequential(
      (0): Identity()
      (1): Linear(in_features=6, out_features=128, bias=True)
      (2): LeakyReLU(negative_slope=0.1)
      (3): Identity()
      (4): Linear(in_features=128, out_features=128, bias=True)
      (5): LeakyReLU(negative_slope=0.1)
    )
    (solvers): ModuleList(
      (0-2): 3 x SubsetSolver(
        (model): Sequential(
          (0): Identity()
          (1): Linear(in_features=256, out_features=128, bias=True)
          (2): LeakyReLU(negative_slope=0.1)
          (3): LinearBlock(
            (norm_layer): Identity()
            (linear): Linear(in_features=128, out_features=128, bias=True)
            (activ): LeakyReLU(negative_slope=0.1)
          )
          (4): LinearBlock(
            (norm_layer): Identity()
            (linear): Linear(in_features=128, out_features=128, bias=True)
            (activ): LeakyReLU(negative_slope=0.1)
          )
          (5): LinearBlock(
      

In [6]:
device = torch.device("cpu")
torch.backends.quantized.engine = "fbgemm"
print("Current quantized engine:", torch.backends.quantized.engine)

# 1. Set a quantization configuration
model.qconfig = tq.get_default_qconfig('fbgemm')
# print("QConfig:", model.qconfig)

# 2. Prepare the model for static quantization
tq.prepare(model, inplace=True)

# 3. Calibration with data
num_cali_events = 200
with torch.no_grad():
    for event_id, event in enumerate(dataloader):
        if event_id == num_cali_events:
            break
        
        readout = event[f'readout_curr_cont'].to(device)
        readout = torch.transpose(readout, 1, 2).flatten(-2, -1)
        
        model(readout)

# 4. Convert to quantized model
quantized_model = tq.convert(model, inplace=False)
print("Quantized model:", quantized_model)

Current quantized engine: fbgemm


/home/yhuang2/anaconda3/envs/rtdc/lib/python3.13/site-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


Quantized model: QuantModel(
  (quant): Quantize(scale=tensor([0.7839]), zero_point=tensor([64]), dtype=torch.quint8)
  (model): MLP(
    (embed): Sequential(
      (0): Identity()
      (1): QuantizedLinear(in_features=6, out_features=128, scale=0.4180126488208771, zero_point=64, qscheme=torch.per_channel_affine)
      (2): QuantizedLeakyReLU(negative_slope=0.1)
      (3): Identity()
      (4): QuantizedLinear(in_features=128, out_features=128, scale=0.23009943962097168, zero_point=82, qscheme=torch.per_channel_affine)
      (5): QuantizedLeakyReLU(negative_slope=0.1)
    )
    (solvers): ModuleList(
      (0): SubsetSolver(
        (model): Sequential(
          (0): Identity()
          (1): QuantizedLinear(in_features=256, out_features=128, scale=0.19832651317119598, zero_point=64, qscheme=torch.per_channel_affine)
          (2): QuantizedLeakyReLU(negative_slope=0.1)
          (3): LinearBlock(
            (norm_layer): Identity()
            (linear): QuantizedLinear(in_features=

In [11]:
stat = {'max_diff': [], 
        'l1_error': [], 
        'l1_norm': []}

with torch.no_grad():
    for event in tqdm(dataloader_test):
        
        readout = event[f'readout_curr_cont'].to(device)
        readout = torch.transpose(readout, 1, 2).flatten(-2, -1)
        
        output_full = model(readout)
        
        output_int8 = quantized_model(readout)

        diff = torch.abs(output_full - output_int8)
        stat['max_diff'].append(diff.max().item())
        stat['l1_error'].append(diff.mean().item())
        stat['l1_norm'].append(output_full.abs().mean().item())

df = pd.DataFrame(data=stat)
df.to_csv('results/float_int8_diff.csv', index=False)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [03:15<00:00, 51.03it/s]


## To ONNX

In [15]:
event = next(iter(dataloader))
readout = event[f'readout_curr_cont'].to(device)
readout = torch.transpose(readout, 1, 2).flatten(-2, -1)

onnx_fname = onnx_folder/"mlp_int8.onnx"

torch.onnx.export(
    quantized_model,
    readout,
    onnx_fname,
    opset_version=13,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
)

onnx_model = onnx.load(onnx_fname )
onnx.checker.check_model(onnx_model)
print("ONNX OK!")

Torch IR graph at exception: graph(%x : Float(4, 50, 6, strides=[300, 6, 1], requires_grad=0, device=cpu),
      %model.embed.1._packed_params._packed_params : __torch__.torch.classes.quantized.LinearPackedParamsBase,
      %model.embed.4._packed_params._packed_params : __torch__.torch.classes.quantized.LinearPackedParamsBase,
      %model.solvers.0.model.1._packed_params._packed_params : __torch__.torch.classes.quantized.LinearPackedParamsBase,
      %model.solvers.0.model.3.linear._packed_params._packed_params : __torch__.torch.classes.quantized.LinearPackedParamsBase,
      %model.solvers.0.model.4.linear._packed_params._packed_params : __torch__.torch.classes.quantized.LinearPackedParamsBase,
      %model.solvers.0.model.5.linear._packed_params._packed_params : __torch__.torch.classes.quantized.LinearPackedParamsBase,
      %model.solvers.1.model.1._packed_params._packed_params : __torch__.torch.classes.quantized.LinearPackedParamsBase,
      %model.solvers.1.model.3.linear._packed

RuntimeError: 0 INTERNAL ASSERT FAILED at "/pytorch/torch/csrc/jit/ir/ir.cpp":579, please report a bug to PyTorch. 173 not in scope